In [ ]:
UPDATE = False
DROP_VIX = False
DROP_SP500 = True
STATA = False
SENTIMENT = True
PresentDate = "2022-06-01"

In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# Preparation of Data Frames

In [ ]:
df  = pd.DataFrame()
# Reading the saved data pickle file

df_sentiment = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment.pkl', compression=None)

In [ ]:
df_dates = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl')

In [ ]:
df_sentiment["Articles"]#.columns.values

## Sentiment

In [ ]:
df_sentiment.rename(columns={"Sentiment_orig":"SentimentOrig",
                "GI_Sentiment_orig":"GI_SentimentOrig"}
                ,inplace=True)
df_sentiment['WordCount'] = df_sentiment['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text

In [ ]:
df_sentiment["PressReleaseCount"].sum()

In [ ]:
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.drop(columns=["Company"], inplace=True)
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
# df_sentiment = df_sentiment[df_sentiment["PressReleaseCount"]>0]
df_sentiment['Sentiment'] = df_sentiment['Sentiment'].apply(np.float)
df_sentiment['SentimentOrig'] = df_sentiment['SentimentOrig'].apply(np.float)
df_sentiment['pos'] = df_sentiment['pos'].apply(np.float)
df_sentiment['neg'] = df_sentiment['neg'].apply(np.float)
df_sentiment['neu'] = df_sentiment['neu'].apply(np.float)
df_sentiment['compound'] = df_sentiment['compound'].apply(np.float)
df_sentiment["polarity"] = df_sentiment["polarity"].apply(np.float64)
df_sentiment["subjectivity"] = df_sentiment["subjectivity"].apply(np.float64)
df_sentiment["GI_Sentiment"] = df_sentiment["GI_Sentiment"].apply(np.float64)
df_sentiment["GI_SentimentOrig"] = df_sentiment["GI_SentimentOrig"].apply(np.float64)
df_sentiment["GI_Positive"] = df_sentiment["GI_Positive"].apply(np.float64)
df_sentiment["GI_Negative"] = df_sentiment["GI_Negative"].apply(np.float64)
df_sentiment["GI_Polarity"] = df_sentiment["GI_Polarity"].apply(np.float64)
df_sentiment["GI_Subjectivity"] = df_sentiment["GI_Subjectivity"].apply(np.float64)
df_sentiment = df_sentiment.drop(df_sentiment.loc[df_sentiment['Sentiment']==0].index)

df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment"].shift(periods=1)
df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment"].shift(periods=2)
df_sentiment["SentimentOrig(t-1)"] = df_sentiment["SentimentOrig"].shift(periods=1)
df_sentiment["SentimentOrig(t-2)"] = df_sentiment["SentimentOrig"].shift(periods=2)
df_sentiment["pos(t-1)"] = df_sentiment["pos"].shift(periods=1)
df_sentiment["pos(t-2)"] = df_sentiment["pos"].shift(periods=2)
df_sentiment["neg(t-1)"] = df_sentiment["neg"].shift(periods=1)
df_sentiment["neg(t-2)"] = df_sentiment["neg"].shift(periods=2)
df_sentiment["neu(t-1)"] = df_sentiment["neu"].shift(periods=1)
df_sentiment["neu(t-2)"] = df_sentiment["neu"].shift(periods=2)
df_sentiment["compound(t-1)"] = df_sentiment["compound"].shift(periods=1)
df_sentiment["compound(t-2)"] = df_sentiment["compound"].shift(periods=2)
df_sentiment["GI_Sentiment(t-1)"] = df_sentiment["GI_Sentiment"].shift(periods=1)
df_sentiment["GI_Sentiment(t-2)"] = df_sentiment["GI_Sentiment"].shift(periods=2)
df_sentiment["GI_SentimentOrig(t-1)"] = df_sentiment["GI_SentimentOrig"].shift(periods=1)
df_sentiment["GI_SentimentOrig(t-2)"] = df_sentiment["GI_SentimentOrig"].shift(periods=2)
df_sentiment["GI_Positive(t-1)"] = df_sentiment["GI_Positive"].shift(periods=1)
df_sentiment["GI_Positive(t-2)"] = df_sentiment["GI_Positive"].shift(periods=2)
df_sentiment["GI_Negative(t-1)"] = df_sentiment["GI_Negative"].shift(periods=1)
df_sentiment["GI_Negative(t-2)"] = df_sentiment["GI_Negative"].shift(periods=2)
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount"].shift(periods=1)
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount"].shift(periods=2)
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount"].shift(periods=1)
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount"].shift(periods=2)
df_sentiment["MarketCap(t-1)"] = df_sentiment["MarketCap"].shift(periods=1)
df_sentiment["MarketCap(t-2)"] = df_sentiment["MarketCap"].shift(periods=2)
df_sentiment["MarketCapFactor(t-1)"] = df_sentiment["MarketCapFactor"].shift(periods=1)
df_sentiment["MarketCapFactor(t-2)"] = df_sentiment["MarketCapFactor"].shift(periods=2)
df_sentiment["GI_Polarity(t-1)"] = df_sentiment["GI_Polarity"].shift(periods=1)
df_sentiment["GI_Subjectivity(t-1)"] = df_sentiment["GI_Subjectivity"].shift(periods=1)


df_sentiment["Sentiment_ln_return"]=np.log(df_sentiment["Sentiment"]/df_sentiment["Sentiment(t-1)"])
df_sentiment["Sentiment_ln_return(t-1)"]=df_sentiment["Sentiment_ln_return"].shift(periods=1)#np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])
df_sentiment["SentimentOrig_ln_return"]=np.log(df_sentiment["SentimentOrig"]/df_sentiment["SentimentOrig(t-1)"])
df_sentiment["SentimentOrig_ln_return(t-1)"]=df_sentiment["SentimentOrig_ln_return"].shift(periods=1)#np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])
df_sentiment["Sentiment_ewm10"]=df_sentiment["Sentiment"].ewm(10).mean()
df_sentiment["Sentiment_ewm20"]=df_sentiment["Sentiment"].ewm(20).mean()
df_sentiment["pos_ln_return"]=np.log(df_sentiment["pos"]/df_sentiment["pos(t-1)"])
df_sentiment["pos_ln_return(t-1)"]=df_sentiment["pos_ln_return"].shift(periods=1)#np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])
df_sentiment["neg_ln_return"]=np.log(df_sentiment["neg"]/df_sentiment["neg(t-1)"])
df_sentiment["neg_ln_return(t-1)"]=df_sentiment["neg_ln_return"].shift(periods=1)#np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])
df_sentiment["neu_ln_return"]=np.log(df_sentiment["neu"]/df_sentiment["neu(t-1)"])
df_sentiment["neu_ln_return(t-1)"]=np.log(df_sentiment["neu(t-1)"]/df_sentiment["neu(t-2)"])
df_sentiment["compound_ln_return"]=np.log(df_sentiment["compound"]/df_sentiment["compound(t-1)"])
df_sentiment["compound_ln_return(t-1)"]=df_sentiment["compound_ln_return"].shift(periods=1)#np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])
df_sentiment["polarity(t-1)"] = df_sentiment["polarity"].shift(periods=1)
df_sentiment["subjectivity(t-1)"] = df_sentiment["subjectivity"].shift(periods=1)
df_sentiment["PressReleaseCount_ewm10"] = df_sentiment["PressReleaseCount"].ewm(10).mean()
df_sentiment["PressReleaseCount_ln_return"]=np.log(df_sentiment["PressReleaseCount"]/df_sentiment["PressReleaseCount(t-1)"])
df_sentiment["PressReleaseCount_ln_return(t-1)"]=df_sentiment["PressReleaseCount_ln_return"].shift(periods=1)#np.log(df_sentiment["PressReleaseCount(t-1)"]/df_sentiment["PressReleaseCount(t-2)"])
df_sentiment["WordCount_ln_return"]=np.log(df_sentiment["WordCount"]/df_sentiment["WordCount(t-1)"])
df_sentiment["WordCount_ln_return(t-1)"]=df_sentiment["WordCount_ln_return"].shift(periods=1)#np.log(df_sentiment["WordCount(t-1)"]/df_sentiment["WordCount(t-2)"])
df_sentiment["GI_Sentiment_ln_return"]=np.log(df_sentiment["GI_Sentiment"]/df_sentiment["GI_Sentiment(t-1)"])
df_sentiment["GI_Sentiment_ln_return(t-1)"]=np.log(df_sentiment["GI_Sentiment(t-1)"]/df_sentiment["GI_Sentiment(t-2)"])
df_sentiment["GI_SentimentOrig_ln_return"]=np.log(df_sentiment["GI_SentimentOrig"]/df_sentiment["GI_SentimentOrig(t-1)"])
df_sentiment["GI_SentimentOrig_ln_return(t-1)"]=np.log(df_sentiment["GI_SentimentOrig(t-1)"]/df_sentiment["GI_SentimentOrig(t-2)"])
df_sentiment["GI_Sentiment_ewm10"]=df_sentiment["GI_Sentiment"].ewm(10).mean()
df_sentiment["GI_Sentiment_ewm20"]=df_sentiment["GI_Sentiment"].ewm(20).mean()
df_sentiment["GI_Positive_ln_return"]=np.log(df_sentiment["GI_Positive"]/df_sentiment["GI_Positive(t-1)"])
df_sentiment["GI_Positive_ln_return(t-1)"]=np.log(df_sentiment["GI_Positive(t-1)"]/df_sentiment["GI_Positive(t-2)"])
df_sentiment["GI_Negative_ln_return"]=np.log(df_sentiment["GI_Negative"]/df_sentiment["GI_Negative(t-1)"])
df_sentiment["GI_Negative_ln_return(t-1)"]=np.log(df_sentiment["GI_Negative(t-1)"]/df_sentiment["GI_Negative(t-2)"])
df_sentiment["MarketCap Natural Log"] = np.log(df_sentiment["MarketCap"]/df_sentiment["MarketCap(t-1)"])
df_sentiment["MarketCap_ln_return(t-1)"] = np.log(df_sentiment["MarketCap(t-1)"]/df_sentiment["MarketCap(t-2)"])
df_sentiment["MarketCapFactor_ln_return"]=np.log(df_sentiment["MarketCapFactor"]/df_sentiment["MarketCapFactor(t-1)"])
df_sentiment["MarketCapFactor_ln_return(t-1)"]=df_sentiment["MarketCapFactor_ln_return"].shift(periods=1)#np.log(df_sentiment["MarketCapFactor(t-1)"]/df_sentiment["MarketCapFactor(t-2)"])

In [ ]:
if (SENTIMENT):

    df_sentiment.rename(columns={"Sentiment_ln_return":"Sentiment Natural Log",
                    "Sentiment_ln_return(t-1)":"Sentiment Natural Log(t-1)",
                    "SentimentOrig_ln_return":"SentimentOrig Natural Log",
                    "SentimentOrig_ln_return(t-1)":"SentimentOrig Natural Log(t-1)",
                    "pos":"Positive",
                    "pos(t-1)":"Positive(t-1)",
                    "pos(t-2)":"Positive(t-2)",
                    "pos_ln_return":"Positive Natural Log",
                    "pos_ln_return(t-1)":"Positive Natural Log(t-1)",
                    "neg":"Negative",
                    "neg(t-1)":"Negative(t-1)", 
                    "neg(t-2)":"Negative(t-2)",
                    "neg_ln_return":"Negative Natural Log", 
                    "neg_ln_return(t-1)":"Negative Natural Log(t-1)", 
                    "neu":"Neutral", 
                    "neu(t-1)":"Neutral(t-1)",
                    "neu(t-2)":"Neutral(t-2)",
                    "neu_ln_return":"Neutral Natural Log",
                    "neu_ln_return(t-1)":"Neutral Natural Log(t-1)",
                    "compound":"Compound",
                    "compound(t-1)":"Compound(t-1)",
                    "compound(t-2)":"Compound(t-2)",
                    "compound_ln_return":"Compound Natural Log",
                    "compound_ln_return(t-1)":"Compound Natural Log(t-1)",
                    "polarity":"Polarity",
                    "polarity(t-1)":"Polarity(t-1)",
                    "polarityB":"PolarityB",
                    "subjectivity":"Subjectivity",
                    "subjectivity(t-1)":"Subjectivity(t-1)",
                    "NewsCount_ln_return":"NewsCount Natural Log",
                    "NewsCount_ln_return(t-1)":"NewsCount Natural Log(t-1)",
                    "WordCount_ln_return":"WordCount Natural Log",
                    "WordCount_ln_return(t-1)":"WordCount Natural Log(t-1)",
                    "MarketCap_ln_return":"MarketCap Natural Log",
                    "MarketCap_ln_return(t-1)":"MarketCap Natural Log(t-1)",
                    "MarketCapFactor_ln_return":"MarketCapFactor Natural Log",
                    "MarketCapFactor_ln_return(t-1)":"MarketCapFactor Natural Log(t-1)",
                    "PressReleaseCount_ln_return":"PressReleaseCount Natural Log",
                    "GI_Sentiment_ln_return":"GI_Sentiment Natural Log",
                    "GI_Sentiment_ln_return(t-1)":"GI_Sentiment Natural Log(t-1)",
                    "GI_SentimentOrig_ln_return":"GI_SentimentOrig Natural Log",
                    "GI_SentimentOrig_ln_return(t-1)":"GI_SentimentOrig Natural Log(t-1)",
                    "GI_Negative_ln_return":"GI_Negative Natural Log",
                    "GI_Negative_ln_return(t-1)":"GI_Negative Natural Log(t-1)",
                    "GI_Positive_ln_return":"GI_Positive Natural Log",
                    "GI_Positive_ln_return(t-1)":"GI_Positive Natural Log(t-1)"}
                    ,inplace=True)

In [ ]:
df_sentiment["PressReleaseCount"]["2020-06-01":]#.sum()

## VIX

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_vix = get_data("^VIX", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")
    df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

In [ ]:
df_vix

In [ ]:
df_vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)

df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
if (DROP_VIX==True):
    df_vix.drop(columns=["high", "low","close", "adjclose", "ticker", "volume"], inplace=True)

df_vix["vix(t-1)"] = df_vix["vix"].shift(periods=1)
df_vix["vix(t-2)"] = df_vix["vix"].shift(periods=2)
df_vix["vix_ln_return"]=np.log(df_vix["vix"]/df_vix["vix(t-1)"])
df_vix["vix_ln_return(t-1)"]=np.log(df_vix["vix(t-1)"]/df_vix["vix(t-2)"])
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

## S&P500

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")
    df_sp500.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl")

In [ ]:
df_sp500=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)

df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
if (DROP_SP500==True):
    df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500(t-1)"] = df_sp500["sp500"].shift(periods=1)
df_sp500["sp500(t-2)"] = df_sp500["sp500"].shift(periods=2)

# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500(t-1)"])/df_sp500["sp500(t-1)"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500(t-1)"])

df_sp500["sp500_ln_return(t-1)"]=np.log(df_sp500["sp500(t-1)"]/df_sp500["sp500(t-2)"])

# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()


# MERGING 

In [ ]:
df = pd.merge(df_sentiment,df_vix, on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

df = pd.merge(df, df_sp500, on='Date')#,how='outer'
df = df.ffill()
df = df.sort_values(by="Date")

# DAY MONTH YEAR

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]
df['quarter'] = pd.PeriodIndex(df.Date, freq='Q')

# THREE PERIODS

In [ ]:
df = df.ffill()
df.index = df.Date
df.ffill(inplace=True)

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':PresentDate]
df_sentiment = df_sentiment.loc['2020-06-01':PresentDate]
df_vix = df_vix.loc['2020-06-01':PresentDate]

In [ ]:
df_part3["Sentiment Natural Log"]

In [ ]:
mx = df_part3.loc[df_part3['Sentiment Natural Log'] != np.inf, 'Sentiment Natural Log'].max()
df_part3['Sentiment Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Sentiment Natural Log'] != -np.inf, 'Sentiment Natural Log'].min()
df_part3['Sentiment Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['SentimentOrig Natural Log'] != np.inf, 'SentimentOrig Natural Log'].max()
df_part3['SentimentOrig Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['SentimentOrig Natural Log'] != -np.inf, 'SentimentOrig Natural Log'].min()
df_part3['SentimentOrig Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['MarketCap Natural Log'] != np.inf, 'MarketCap Natural Log'].max()
df_part3['MarketCap Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['MarketCap Natural Log'] != -np.inf, 'MarketCap Natural Log'].min()
df_part3['MarketCap Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['Sentiment Natural Log(t-1)'] != np.inf, 'Sentiment Natural Log(t-1)'].max()
df_part3['Sentiment Natural Log(t-1)'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Sentiment Natural Log(t-1)'] != -np.inf, 'Sentiment Natural Log(t-1)'].min()
df_part3['Sentiment Natural Log(t-1)'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['Negative Natural Log'] != np.inf, 'Negative Natural Log'].max()
df_part3['Negative Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Negative Natural Log'] != -np.inf, 'Negative Natural Log'].min()
df_part3['Negative Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['Neutral Natural Log'] != np.inf, 'Neutral Natural Log'].max()
df_part3['Neutral Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Neutral Natural Log'] != -np.inf, 'Neutral Natural Log'].min()
df_part3['Neutral Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['Compound Natural Log'] != np.inf, 'Compound Natural Log'].max()
df_part3['Compound Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Compound Natural Log'] != -np.inf, 'Compound Natural Log'].min()
df_part3['Compound Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['Positive Natural Log'] != np.inf, 'Positive Natural Log'].max()
df_part3['Positive Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Positive Natural Log'] != -np.inf, 'Positive Natural Log'].min()
df_part3['Positive Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['Negative Natural Log'] != np.inf, 'Negative Natural Log'].max()
df_part3['Negative Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Negative Natural Log'] != -np.inf, 'Negative Natural Log'].min()
df_part3['Negative Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['Neutral Natural Log'] != np.inf, 'Neutral Natural Log'].max()
df_part3['Neutral Natural Log'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Neutral Natural Log'] != -np.inf, 'Neutral Natural Log'].min()
df_part3['Neutral Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Compound Natural Log'] != np.inf, 'Compound Natural Log'].max()
df['Compound Natural Log'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Compound Natural Log'] != -np.inf, 'Compound Natural Log'].min()
df['Compound Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['GI_Sentiment Natural Log'] != np.inf, 'GI_Sentiment Natural Log'].max()
df['GI_Sentiment Natural Log'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['GI_Sentiment Natural Log'] != -np.inf, 'GI_Sentiment Natural Log'].min()
df['GI_Sentiment Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['GI_SentimentOrig Natural Log'] != np.inf, 'GI_SentimentOrig Natural Log'].max()
df['GI_SentimentOrig Natural Log'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['GI_SentimentOrig Natural Log'] != -np.inf, 'GI_SentimentOrig Natural Log'].min()
df['GI_SentimentOrig Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['GI_Positive Natural Log'] != np.inf, 'GI_Positive Natural Log'].max()
df['GI_Positive Natural Log'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['GI_Positive Natural Log'] != -np.inf, 'GI_Positive Natural Log'].min()
df['GI_Positive Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['GI_Negative Natural Log'] != np.inf, 'GI_Negative Natural Log'].max()
df['GI_Negative Natural Log'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['GI_Negative Natural Log'] != -np.inf, 'GI_Negative Natural Log'].min()
df['GI_Negative Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['GI_Polarity'] != np.inf, 'GI_Polarity'].max()
df['GI_Polarity'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['GI_Polarity'] != -np.inf, 'GI_Polarity'].min()
df['GI_Polarity'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['GI_Subjectivity'] != np.inf, 'GI_Subjectivity'].max()
df['GI_Subjectivity'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['GI_Subjectivity'] != -np.inf, 'GI_Subjectivity'].min()
df['GI_Subjectivity'].replace(-np.inf,mn,inplace=True)

In [ ]:
mn = df_part3.loc[df_part3['Sentiment Natural Log'] != -np.inf, 'Sentiment Natural Log'].min()
df_part3['Sentiment Natural Log'] = df_part3['Sentiment Natural Log'].fillna(mn)

mn = df_part3.loc[df_part3['vix_ln_return'] != -np.inf, 'vix_ln_return'].min()
df_part3['vix_ln_return'] = df_part3['vix_ln_return'].fillna(mn)

mn = df_part3.loc[df_part3['sp500_ln_return'] != -np.inf, 'sp500_ln_return'].min()
df_part3['sp500_ln_return'] = df_part3['sp500_ln_return'].fillna(mn)

mn = df_part3.loc[df_part3['PressReleaseCount Natural Log'] != -np.inf, 'PressReleaseCount Natural Log'].min()
df_part3['PressReleaseCount Natural Log'] = df_part3['PressReleaseCount Natural Log'].fillna(mn)

In [ ]:
# df = pd.DataFrame()
# for i in (range(24)):

#     # df["Sentiment Natural Log"].ffill(inplace=True)
#     df["Sentiment Natural Log"] = df.loc[df["Sentiment Natural Log"]!=-np.inf]
#     df["Sentiment Natural Log"] = df.loc[df["Sentiment Natural Log"]!=np.inf]

#     df["Sentiment Natural Log"] = df["Sentiment Natural Log"].replace(np.inf,0)

#     df["Sentiment Natural Log"] = df["Sentiment Natural Log"].replace(-np.inf,0)

In [ ]:
# df_part3.set_index(['Date', 'Year', 'Month', 'Day'], inplace=True)

In [ ]:
df_part3.loc[df_part3["PressReleaseCount"]>0]

In [ ]:
df_part3['Articles'].apply(lambda x:len(str(x).split())).sum() #Number Of words in main text

In [ ]:
df_part3[["vix_ln_return","Sentiment Natural Log","Sentiment"]].sort_values(by="vix_ln_return")

In [ ]:
df.drop(columns={"Date"},inplace=True)#(inplace=True)

In [ ]:
df["Date"] = df.index

In [ ]:
df.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "vix_ln_return":"VIX Natural Log",
                "sp500":"S&P500",
                "sp500(t-1)":"S&P500(t-1)",
                "sp500(t-2)":"S&P500(t-2)",
                "sp500_ln_return":"S&P500 Natural Log",
                "Sentiment_ln_return":"Sentiment Natural Log",
                "Sentiment_ln_return(t-1)":"Sentiment Natural Log(t-1)",
                "SentimentOrig_ln_return":"SentimentOrig Natural Log",
                "SentimentOrig_ln_return(t-1)":"SentimentOrig Natural Log(t-1)",
                "pos":"Positive",
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)",
                "pos_ln_return":"Positive Natural Log",
                "pos_ln_return(t-1)":"Positive Natural Log(t-1)",
                "neg":"Negative",
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)",
                "neg_ln_return":"Negative Natural Log", 
                "neg_ln_return(t-1)":"Negative Natural Log(t-1)", 
                "neu":"Neutral", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "neu_ln_return":"Neutral Natural Log",
                "neu_ln_return(t-1)":"Neutral Natural Log(t-1)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound Natural Log",
                "compound_ln_return(t-1)":"Compound Natural Log(t-1)",
                "polarity":"Polarity",
                "polarity(t-1)":"Polarity(t-1)",
                "polarityB":"PolarityB",
                "subjectivity":"Subjectivity",
                "subjectivity(t-1)":"Subjectivity(t-1)",
                "PressReleaseCount_ln_return":"PressReleaseCount Natural Log",
                "PressReleaseCount_ln_return(t-1)":"PressReleaseCount Natural Log(t-1)",
                "WordCount_ln_return":"WordCount Natural Log",
                "WordCount_ln_return(t-1)":"WordCount Natural Log(t-1)",
                "MarketCap_ln_return":"MarketCap Natural Log",
                "MarketCap_ln_return(t-1)":"MarketCap Natural Log(t-1)",
                "MarketCapFactor_ln_return":"MarketCapFactor Natural Log",
                "MarketCapFactor_ln_return(t-1)":"MarketCapFactor Natural Log(t-1)",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment Natural Log",
                "GI_Sentiment_ln_return(t-1)":"GI_Sentiment Natural Log(t-1)",
                "GI_SentimentOrig_ln_return":"GI_SentimentOrig Natural Log",
                "GI_SentimentOrig_ln_return(t-1)":"GI_SentimentOrig Natural Log(t-1)",
                "GI_Negative_ln_return":"GI_Negative Natural Log",
                "GI_Negative_ln_return(t-1)":"GI_Negative Natural Log(t-1)",
                "GI_Positive_ln_return":"GI_Positive Natural Log",
                "GI_Positive_ln_return(t-1)":"GI_Positive Natural Log(t-1)"}
                ,inplace=True)

In [ ]:
df_full = df
df = df.loc['2020-06-01':PresentDate]

# df_part

In [ ]:
df["PressReleaseCount"]["2020-06-01":].sum()

# STATA

In [ ]:
# STATA = True

In [ ]:
if (STATA):
    # df = df.round(2)
    # df.dropna(inplace=True)
    df.rename(columns={"VIX":"vix",
                "VIX(t-1)":"vix(t-1)",
                "VIX Natural Log":"vix_ln_return",
                "VIX Natural Log(t-1)":"vix_ln_return_t_1",
                "S&P500":"sp500",
                "S&P500 Natural Log":"sp500_ln_return",
                "S&P500 Natural Log(t-1)":"sp500_ln_return_t_1",
                "Sentiment Natural Log":"sentiment_ln_return",
                "Sentiment Natural Log(t-1)":"sentiment_ln_return_lag1",
                "Positive":"pos",
                "Positive(t-1)":"pos(t-1)",
                "Positive(t-2)":"pos(t-2)",
                "Positive Natural Log":"pos_ln_return",
                "Positive Natural Log(t-1)":"pos_ln_return(t-1)",
                "Negative":"neg",
                "Negative(t-1)":"neg(t-1)", 
                "Negative(t-2)":"neg(t-2)",
                "Negative Natural Log":"neg_ln_return", 
                "Negative Natural Log(t-1)":"neg_ln_return(t-1)", 
                "Neutral":"neu", 
                "Neutral(t-1)":"neu(t-1)",
                "Neutral(t-2)":"neu(t-2)",
                "Neutral Natural Log":"neu_ln_return",
                "Neutral Natural Log(t-1)":"neu_ln_return(t-1)",
                "Compound":"compound",
                "Compound(t-1)":"compound(t-1)",
                "Compound(t-2)":"compound(t-2)",
                "Compound Natural Log":"compound_ln_return",
                "Compound Natural Log(t-1)":"compound_ln_return(t-1)",
                "Polarity":"polarity",
                "Polarity(t-1)":"polarity(t-1)",
                "PolarityB":"polarityB",
                "Subjectivity":"subjectivity",
                "Subjectivity(t-1)":"subjectivity(t-1)",
                "PressReleaseCount Natural Log":"PressReleaseCount_ln_return",
                "PressReleaseCount Natural Log(t-1)":"PressReleaseCount_ln_return(t-1)",
                "WordCount Natural Log":"WordCount_ln_return",
                "WordCount Natural Log(t-1)":"WordCount_ln_return(t-1)",
                "Day of Week":"day",
                "Month":"month",
                "Month-Year":"month-year",
                "Quarter":"quarter",
                "Year":"year",
                "GI_Sentiment Natural Log":"GI_Sentiment_ln_return",
                "GI_SentimentOrig Natural Log":"GI_SentimentOrig_ln_return",
                "GI_Negative Natural Log":"GI_Negative_ln_return",
                "GI_Positive Natural Log":"GI_Positive_ln_return"}
                ,inplace=True)
    df.to_csv("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/stata/Sentiment.csv")
    print("STATA FILE CREATED")

In [ ]:
if (SENTIMENT):
    import datetime as dt
    from datetime import date
    import calendar
    my_date = date.today()
    calendar.day_name[my_date.weekday()]  #'Wednesday'

    # Prepare data
    df_sentiment['year'] = [d.year for d in df_sentiment.Date]
    df_sentiment['month'] = [d.strftime('%b') for d in df_sentiment.Date]
    df_sentiment['day'] = [d.day_name() for d in df_sentiment.Date]
    years = df_sentiment['year'].unique()
    df_sentiment['month-year'] = [d.strftime('%b') + str(d.year) for d in df_sentiment.Date]
    df_sentiment['quarter'] = pd.PeriodIndex(df_sentiment.Date, freq='Q')

    df_sentiment.rename(columns={
                    "day":"Day of Week",
                    "month":"Month",
                    "month-year":"Month-Year",
                    "quarter":"Quarter",
                    "year":"Year"}
                    ,inplace=True)

In [ ]:
if (SENTIMENT):
    import datetime as dt
    from datetime import date
    import calendar
    my_date = date.today()
    calendar.day_name[my_date.weekday()]  #'Wednesday'

    # Prepare data
    df_sp500['year'] = [d.year for d in df_sp500.Date]
    df_sp500['month'] = [d.strftime('%b') for d in df_sp500.Date]
    df_sp500['day'] = [d.day_name() for d in df_sp500.Date]
    years = df_sp500['year'].unique()
    df_sp500['month-year'] = [d.strftime('%b') + str(d.year) for d in df_sp500.Date]
    df_sp500['quarter'] = pd.PeriodIndex(df_sp500.Date, freq='Q')

    df_sp500.rename(columns={"vix":"VIX",
                    "vix(t-1)":"VIX(t-1)",
                    "vix_ln_return":"VIX Natural Log",
                    "sp500":"S&P500",
                    "sp500(t-1)":"S&P500(t-1)",
                    "sp500(t-2)":"S&P500(t-2)",
                    "sp500_ln_return":"S&P500 Natural Log",
                    "day":"Day of Week",
                    "month":"Month",
                    "month-year":"Month-Year",
                    "quarter":"Quarter",
                    "year":"Year"}
                    ,inplace=True)

In [ ]:
if (SENTIMENT):
    import datetime as dt
    from datetime import date
    import calendar
    my_date = date.today()
    calendar.day_name[my_date.weekday()]  #'Wednesday'

    # Prepare data
    df_vix['year'] = [d.year for d in df_vix.Date]
    df_vix['month'] = [d.strftime('%b') for d in df_vix.Date]
    df_vix['day'] = [d.day_name() for d in df_vix.Date]
    years = df_vix['year'].unique()
    df_vix['month-year'] = [d.strftime('%b') + str(d.year) for d in df_vix.Date]
    df_vix['quarter'] = pd.PeriodIndex(df_vix.Date, freq='Q')

    df_vix.rename(columns={"vix":"VIX",
                    "vix(t-1)":"VIX(t-1)",
                    "vix_ln_return":"VIX Natural Log",
                    "sp500":"S&P500",
                    "sp500(t-1)":"S&P500(t-1)",
                    "sp500(t-2)":"S&P500(t-2)",
                    "sp500_ln_return":"S&P500 Natural Log",
                    "day":"Day of Week",
                    "month":"Month",
                    "month-year":"Month-Year",
                    "quarter":"Quarter",
                    "year":"Year"}
                    ,inplace=True)

In [ ]:
# df.drop(columns={"Day of Week"},inplace=True)

In [ ]:
df["VIX Natural Log"].std()#=="Monday"

In [ ]:
df["Sentiment Natural Log"].median()

In [ ]:
df